### Imports

In [1]:
import sys

sys.path.insert(0, "../../")
from config import DATASETS_PATH, IMG_SHAPE, SEED

### Functions

In [2]:
import os
import shutil
from sklearn.model_selection import train_test_split
import pathlib

def split_train_data(train_dir, validation_split=0.15, test_split=0.15, seed=42):
    """
    Splits a training dataset into train, validation, and test sets by moving files.

    The function operates on a directory that contains subfolders for each class.
    It creates 'validation' and 'test' directories at the same level as the
    'train' directory and moves a percentage of images into them. The original
    'train' directory will be modified (files will be removed).

    Args:
        train_dir (str): Path to the training directory (e.g., './data/train').
        validation_split (float): Percentage of data to move to the validation set (e.g., 0.15 for 15%).
        test_split (float): Percentage of data to move to the test set (e.g., 0.15 for 15%).
        seed (int): Random seed for reproducibility of the split.
    """
    # --- 1. Input Validation ---
    assert 0 <= validation_split + test_split < 1, "The sum of splits must be less than 1."
    
    # --- 2. Define Directory Paths ---
    train_dir = pathlib.Path(train_dir)
    root_dir = train_dir.parent
    validation_dir = root_dir / 'validation'
    test_dir = root_dir / 'test'

    print(f"Train directory: {train_dir}")
    print(f"Output validation directory: {validation_dir}")
    print(f"Output test directory: {test_dir}")

    # --- 3. Create Output Directories ---
    os.makedirs(validation_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # --- 4. Iterate Through Classes and Split Data ---
    for class_name in os.listdir(train_dir):
        class_path = train_dir / class_name
        if not os.path.isdir(class_path):
            continue

        print(f"\nProcessing class: {class_name}")

        # Get all image file paths
        images = [f for f in os.listdir(class_path) if os.path.isfile(class_path / f)]
        
        if len(images) < 3:
            print(f"  -> Skipping class '{class_name}', not enough images to split.")
            continue

        # --- First split: Separate a combined validation/test set ---
        # The remaining images will stay in the train folder.
        train_imgs, val_and_test_imgs = train_test_split(
            images,
            test_size=(validation_split + test_split),
            random_state=seed
        )

        # --- Second split: Separate validation and test from the combined set ---
        # We need to calculate the new proportion for the test set relative to the combined set.
        if validation_split > 0 and test_split > 0:
            test_proportion = test_split / (validation_split + test_split)
            validation_imgs, test_imgs = train_test_split(
                val_and_test_imgs,
                test_size=test_proportion,
                random_state=seed
            )
        elif validation_split > 0:
            validation_imgs = val_and_test_imgs
            test_imgs = []
        else: # test_split must be > 0
            test_imgs = val_and_test_imgs
            validation_imgs = []

        # --- 5. Create Class Subfolders in Destination and Move Files ---
        val_class_dir = validation_dir / class_name
        test_class_dir = test_dir / class_name
        os.makedirs(val_class_dir, exist_ok=True)
        os.makedirs(test_class_dir, exist_ok=True)

        # Move validation images
        for img in validation_imgs:
            shutil.move(str(class_path / img), str(val_class_dir / img))
        
        # Move test images
        for img in test_imgs:
            shutil.move(str(class_path / img), str(test_class_dir / img))

        print(f"  -> Original: {len(images)} images")
        print(f"  -> Remaining in train: {len(os.listdir(class_path))}")
        print(f"  -> Moved to validation: {len(validation_imgs)}")
        print(f"  -> Moved to test: {len(test_imgs)}")

    print("\n✅ Splitting complete.")

### VAE Dataset split

Manually copy the classes intented for the vae in a tree structure like the following. In the next steps this notebook will create the test split

```text
|-- classification/
    |-- train/
        |-- tagged/
            |-- prophase/
                |-- example_prophase_image.png
            |-- metaphase/
                |-- example_metaphase_image.png
            |-- anaphase/
                |-- example_anaphase_image.png
            |-- telophase/
                |-- example_telophase_image.png
        |-- untagged/
            |-- example_untagged_image.png
```

In [ ]:
raise ValueError("Please organize images in the classes folders") #Intended to stop the script here to mannualy organize classes

ValueError: Please organize images in the classes folders

#### Creation of train - test split

In [ ]:
VAE_DATASET_PATH = os.path.join(DATASETS_PATH, 'cropped', 'vae')

TRAIN_PATH = os.path.join(VAE_DATASET_PATH,  'train', 'tagged')
TEST_PATH = os.path.join(VAE_DATASET_PATH,  'test')

split_dataset(TRAIN_PATH, TEST_PATH, test_size=0.2)


Class 'interphase': 428 test images copied.
Class 'metaphase': 48 test images copied.
Class 'telophase': 44 test images copied.
Class 'anaphase': 40 test images copied.
Class 'prophase': 55 test images copied.


#### Pending manual task

After the classes directories are created it is needed to augment the images. For that there is a notebook (data_augmentation.ipynb) which will have to be run for the classes and the untagged images three times for each. The prophase folder can be augmented only once since it is the class with the most amount of images

### Classifier split

In [3]:
CLASSIFIER_DATASET_PATH = os.path.join(DATASETS_PATH, 'cropped', 'classifier')

TRAIN_PATH = os.path.join(DATASETS_PATH, 'cropped',  'classifier', 'train')
TEST_PATH = os.path.join(DATASETS_PATH, 'cropped',  'classifier', 'test')

split_train_data(TRAIN_PATH, validation_split=0.15, test_split=0.15, seed=42)

Train directory: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/classifier/train
Output validation directory: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/classifier/validation
Output test directory: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/classifier/test

Processing class: interphase
  -> Original: 2140 images
  -> Remaining in train: 1498
  -> Moved to validation: 321
  -> Moved to test: 321

Processing class: metaphase
  -> Original: 238 images
  -> Remaining in train: 166
  -> Moved to validation: 36
  -> Moved to test: 36

Processing class: telophase
  -> Original: 218 images
  -> Remaining in train: 152
  -> Moved to validation: 33
  -> Moved to test: 33

Processing class: anaphase
  -> Original: 200 images
  -> Remaining in train: 140
  -> Moved to validation: 30
  -> Moved to test: 30

Processing class: prophase
  -> Original: 272 images
  -> Remaining in train: 190
  -> Moved to validation: 41
  -> Moved to test: 41

✅ S